# Silver Layer — Cleaning, Casting & Deduplication
**What this does:** Reads the 3 Bronze tables, cleans them, fixes data types, extracts nested fields, removes duplicates, and writes clean Silver tables ready for Gold/dbt.

**Why Silver exists:** Bronze is raw — types are wrong (timestamps stored as plain text), nested JSON is still locked inside string columns, and duplicate rows can exist if the pipeline ran twice. Silver fixes all of that. Think of it as the X-ray machine that inspects every bag and fixes broken tags before sending them to the next belt.

In [0]:
# functions as F = Spark's (distributed processing engine) built-in column functions
# F.col()               = reference a column by name
# F.col().cast()        = convert a column to a different type (string → timestamp, etc.)
# F.get_json_object()   = reach inside a JSON string and pull out one specific field
# .dropna()             = remove rows where critical columns are null/missing
# .dropDuplicates()     = keep only one row when two rows are identical on specified columns
from pyspark.sql import functions as F

In [0]:
# ── SILVER PRICES ─────────────────────────────────────────────────────────────
# Read the raw Bronze table
df_prices_bronze = spark.table("bronze_prices")

# .select()   = pick only the columns we need and cast them to correct types
# .cast("double")    = convert to decimal number (65777.53 needs decimals, not integer)
# .cast("integer")   = convert to whole number (rank is always 1, 2, 3...)
# .cast("timestamp") = convert text "2026-06-17T03:36:30.999Z" to a real date object
#                      so Spark can sort by it, filter date ranges, do date math
# .dropna()          = remove rows where critical columns are null/missing
#                      a price row with no coin ID or no price is useless garbage
# .dropDuplicates()  = if the same coin appears at the same timestamp twice (pipeline ran twice),
#                      keep only one copy — prevents double-counting in analytics
df_silver_prices = df_prices_bronze.select(
    F.col("id"),
    F.col("symbol"),
    F.col("name"),
    F.col("current_price").cast("double"),
    F.col("market_cap").cast("double"),
    F.col("market_cap_rank").cast("integer"),
    F.col("total_volume").cast("double"),
    F.col("high_24h").cast("double"),
    F.col("low_24h").cast("double"),
    F.col("price_change_24h").cast("double"),
    F.col("price_change_percentage_24h").cast("double"),
    F.col("circulating_supply").cast("double"),
    F.col("last_updated").cast("timestamp")
).dropna(
    subset=["id", "current_price", "last_updated"]
).dropDuplicates(
    ["id", "last_updated"]
)

print(f"Silver prices rows: {df_silver_prices.count()}")
display(df_silver_prices)

In [0]:
# ── SILVER FEAR & GREED ───────────────────────────────────────────────────────
# In Bronze, the actual fear/greed number is locked inside the "data" column as a JSON string:
# {"name": "Fear and Greed Index", "data": [{"value": "22", "value_classification": "Extreme Fear", ...}]}
# Silver's job here: crack open that string and pull out the useful fields.
df_fg_bronze = spark.table("bronze_fear_greed")

df_silver_fg = df_fg_bronze.select(

    # Cast ingested_at from plain text to a real timestamp
    F.col("ingested_at").cast("timestamp").alias("ingested_at"),
    F.col("source"),

    # get_json_object() = reach inside a JSON string column and extract one value
    # "$.data[0].value" means: go into the "data" array, take the first item [0], get "value"
    # WHY [0]? The API returns an array but always has exactly one current reading.
    # Cast to integer because "22" comes in as text — we need it as a real number for analytics
    F.get_json_object(F.col("data"), "$.data[0].value").cast("integer").alias("fear_greed_value"),
    F.get_json_object(F.col("data"), "$.data[0].value_classification").alias("value_classification"),

    # The API's own Unix timestamp (seconds since 1970) — useful for time-travel joins later
    # Unix timestamp = a single big integer representing a point in time, e.g. 1781654400
    F.get_json_object(F.col("data"), "$.data[0].timestamp").cast("long").alias("fg_unix_timestamp")

) \
.dropna(subset=["ingested_at", "fear_greed_value"]) \
.dropDuplicates(["ingested_at"])

print(f"Silver fear & greed rows: {df_silver_fg.count()}")
display(df_silver_fg)

In [0]:
# ── SILVER NEWS ───────────────────────────────────────────────────────────────
# The cryptocurrency.cv free tier returns 0 articles every time (articles: [], totalCount: 0).
# This is a free tier limitation — the API works, but articles require a paid plan.
# We still build the Silver table properly so:
#   1. The pipeline doesn't crash
#   2. When you upgrade to paid, data flows automatically without changing any code
df_news_bronze = spark.table("bronze_news")

df_silver_news = df_news_bronze.select(

    F.col("ingested_at").cast("timestamp").alias("ingested_at"),
    F.col("source"),

    # Extract totalCount to track how many articles came in (currently always 0)
    F.get_json_object(F.col("data"), "$.totalCount").cast("integer").alias("total_article_count"),

    # Keep the raw data string — when articles appear (paid tier), Gold/dbt can parse them
    F.col("data").alias("raw_data")

) \
.dropDuplicates(["ingested_at"])

print(f"Silver news rows: {df_silver_news.count()}")
print("Note: total_article_count will be 0 until you upgrade to cryptocurrency.cv paid tier")
display(df_silver_news)

In [0]:
# ── WRITE TO DELTA LAKE SILVER TABLES ─────────────────────────────────────────
# Same pattern as Bronze: Delta format, overwrite mode, registered as named tables
# The difference from Bronze: these tables are CLEAN — correct types, no nulls, no duplicates
df_silver_prices.write.format("delta").mode("overwrite").saveAsTable("silver_prices")
df_silver_fg.write.format("delta").mode("overwrite").saveAsTable("silver_fear_greed")
df_silver_news.write.format("delta").mode("overwrite").saveAsTable("silver_news")

print("Silver tables written:")
print("  silver_prices")
print("  silver_fear_greed")
print("  silver_news")

In [0]:
# ── VERIFY ────────────────────────────────────────────────────────────────────
display(spark.sql("SHOW TABLES"))